In [2]:
import torch
import csv
import spacy
import re
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchtext.vocab import GloVe

In [3]:
class MovieReviewsDataset(Dataset):
    def __init__(self, dataset_file):
        self.reviews = []
        self.labels = []
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

        with open(dataset_file, newline='', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile, delimiter=',')
            next(reader, None)
            for row in reader:
                self.reviews.append(row[0])
                self.labels.append(1 if row[1] == 'positive' else 0)

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = self.reviews[idx]
        clean_text = re.sub(r"<.*?>", "", review)
        review_doc = self.nlp(clean_text)
        review_doc = [token.lemma_.lower() for token in review_doc if token.is_alpha and not token.is_stop]

        return review_doc, self.labels[idx]
        

In [4]:
class SentimentCNN(nn.Module):
    def __init__(self, embedding_dim, num_classes=2):
        super().__init__()
        
        # 1D CNN layers with different kernel sizes (n-grams)
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=3)
        self.conv2 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=4)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=5)
        
        # Fully connected layer
        self.fc = nn.Linear(3 * 100, num_classes)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # Embed words: (batch_size, seq_len, embedding_dim)
        x = self.embedding(x)
        
        # CNN expects (batch_size, embedding_dim, seq_len)
        x = x.permute(0, 2, 1)
        
        # Apply convolution + ReLU + global max pooling
        x1 = F.relu(self.conv1(x)).max(dim=2)[0]
        x2 = F.relu(self.conv2(x)).max(dim=2)[0]
        x3 = F.relu(self.conv3(x)).max(dim=2)[0]
        
        # Concatenate pooled outputs
        x = torch.cat([x1, x2, x3], dim=1)
        x = self.dropout(x)
        
        # Fully connected layer
        logits = self.fc(x)
        return logits


In [5]:
def collate_fn(batch, max_len=100):
    reviews, labels = zip(*batch)
    reviews_idx = []
    for r in reviews:
        idxs = [vec.stoi.get(tok, 0) for tok in r]
        if len(idxs) < max_len:
            idxs += [0] * (max_len - len(idxs))  # pad
        else:
            idxs = idxs[:max_len]
        reviews_idx.append(idxs)
    reviews_tensor = torch.tensor(reviews_idx, dtype=torch.long)
    labels_tensor = torch.tensor([int(l) for l in labels], dtype=torch.long)
    return reviews_tensor, labels_tensor


In [13]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
df = pd.read_csv(r'..\data\IMDB Dataset.csv')
df



,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only ""has got all the polari"" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.<br /><br />This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson, in this she managed to tone down her ""sexy"" image and jumped right into a average, but spirited young woman.<br /><br />This may not be the crown jewel of his career, but it was wittier than ""Devil Wears Prada"" and more interesting than ""Superman"" a great comedy to go see with friends.",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zom

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mrd = MovieReviewsDataset(r".\data\IMDB Dataset.csv")

g = torch.manual_seed(49)
train_set, validation_set, test_set = random_split(mrd, [0.7, 0.15, 0.15], generator=g)

train_dataloader = DataLoader(train_set, batch_size=128, shuffle=True, collate_fn=collate_fn)
validation_dataloader = DataLoader(validation_set, batch_size=128, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_set, batch_size=128, shuffle=False, collate_fn=collate_fn)

embedding_dim = 300
vec = GloVe(name="6B", dim=embedding_dim)
model = SentimentCNN(embedding_dim, num_classes=2).to(device)
model.embedding = nn.Embedding.from_pretrained(vec.vectors, freeze=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
best_val_loss = float("inf")
save_path = "sentiment_cnn.pt"

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_dataloader, 0):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:  
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i+1):.3f}')
            running_loss = 0.0

    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in validation_dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = val_loss / len(validation_dataloader)
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{epochs} "
          f"Train Loss: {running_loss/len(train_dataloader):.4f} "
          f"Val Loss: {avg_val_loss:.4f} "
          f"Val Acc: {val_acc:.4f}")

    # ---- Save best model ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Model improved. Saved to {save_path}")